# Migration Phase 6: cleanup verification

Phase 6 removes the obsolete pre-tabular-derivatives loaders, dead
transformers, and unused dependencies/`.env` keys:

- **Removed loaders**: `anatomical.py`, `anatomical_example.py`,
  `diffusion.py`, `questionnaire.py` (and their exports
  `AnatomicalLoader`/`AnatomicalPaths`/`DiffusionLoader`/`DiffusionPaths`/
  `QuestionnaireLoader`).
- **`parse_bids_entities`** moved from `diffusion.py` into
  `tabular_derivatives.py` (its only remaining consumer) and is still
  re-exported from `neuroalign.data.loaders`.
- **Removed `transformers.py`** (`AnatomicalWideTransformer` /
  `DiffusionWideTransformer`) - both operated on the obsolete
  pre-Phase-4 long format and are no longer used; `FeatureStore` now
  generates wide features directly.
- **Removed dependencies**: `parcellate` (and its transitive deps
  `pybids`/`bids-validator`/...), `gam`, `pygam` - all unused by the new
  pipeline.
- **Removed `.env` keys**: `QSIPARC_PATH`, `QSIRECON_PATH`, `CAT12_*`,
  `SESSIONS_CSV`, `QUESTIONNAIRE_CSV`, `MATLAB_BIN`, `SPM_PATH`,
  `CAT12_PATH`, `TIV_TEMPLATE`.

This notebook verifies the cleaned-up package still imports and works
end-to-end against the existing Phase 5 demo store.

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv(Path.cwd().parent / ".env")

import neuroalign.data.loaders as loaders
import neuroalign.data.preprocessing as preprocessing

print("loaders.__all__:", loaders.__all__)
print("preprocessing.__all__:", preprocessing.__all__)

loaders.__all__: ['BehavioralLoader', 'TabularDerivativesLoader', 'parse_bids_entities']
preprocessing.__all__: ['PipelineConfig', 'DataPaths', 'ModalityConfig', 'OutputConfig', 'FeatureStore', 'FeatureInfo', 'LongFormatInfo', 'StoreManifest', 'ANATOMICAL_METRICS', 'DataPreparationPipeline', 'PipelineResult']


## 1. Obsolete modules are gone

Importing the removed modules/symbols should fail.

In [2]:
removed_modules = [
    "neuroalign.data.loaders.anatomical",
    "neuroalign.data.loaders.anatomical_example",
    "neuroalign.data.loaders.diffusion",
    "neuroalign.data.loaders.questionnaire",
    "neuroalign.data.preprocessing.transformers",
]

import importlib

for mod in removed_modules:
    try:
        importlib.import_module(mod)
        print(f"STILL PRESENT: {mod}")
    except ModuleNotFoundError:
        print(f"removed: {mod}")

removed_symbols = [
    "AnatomicalLoader",
    "AnatomicalPaths",
    "DiffusionLoader",
    "DiffusionPaths",
    "QuestionnaireLoader",
]
for sym in removed_symbols:
    assert not hasattr(loaders, sym), f"{sym} still exported from loaders"
print("removed symbols are no longer exported from neuroalign.data.loaders")

removed: neuroalign.data.loaders.anatomical
removed: neuroalign.data.loaders.anatomical_example
removed: neuroalign.data.loaders.diffusion
removed: neuroalign.data.loaders.questionnaire
removed: neuroalign.data.preprocessing.transformers
removed symbols are no longer exported from neuroalign.data.loaders


## 2. `parse_bids_entities` survives the move

Now defined in `tabular_derivatives.py`, still re-exported from
`neuroalign.data.loaders` and used by `TabularDerivativesLoader.load_diffusion`.

In [3]:
from neuroalign.data.loaders import parse_bids_entities

parse_bids_entities("sub-001_ses-01_software-DSIStudio_model-tensor_param-fa_diffmap.tsv")

{'sub': '001',
 'ses': '01',
 'software': 'DSIStudio',
 'model': 'tensor',
 'param': 'fa'}

## 3. Dependency cleanup

`uv sync --all-extras` after the `pyproject.toml` edit removed `parcellate`
(and its transitive deps `pybids`, `bids-validator`, ...), `gam`, and
`pygam` from the environment.

In [4]:
import importlib.util

for pkg in ["parcellate", "pygam", "gam", "pybids"]:
    spec = importlib.util.find_spec(pkg)
    print(f"{pkg}: {'still installed' if spec else 'not installed'}")

parcellate: not installed
pygam: not installed
gam: not installed
pybids: not installed


## 4. End-to-end pipeline still works

Re-run the Phase 5 demo store (incremental - everything already loaded).

In [5]:
from neuroalign.data.preprocessing import (
    PipelineConfig,
    DataPaths,
    ModalityConfig,
    DataPreparationPipeline,
)

store_dir = Path.cwd().parent / "data" / "processed" / "phase5_demo"

config = PipelineConfig(
    paths=DataPaths(
        brainlink_db=os.environ["BRAINLINK_DB_PATH"],
        tabular_derivatives_root=os.environ["TABULAR_DERIVATIVES_ROOT"],
        output_dir=store_dir,
    ),
    modalities=ModalityConfig(anatomical=True, diffusion=True),
    atlas_name=os.environ["ATLAS_NAME"],
    anat_atlases=tuple(os.environ["ANAT_ATLASES"].split(",")),
    session_variant=os.environ["SESSION_VARIANT"],
    labs=["TS"],
)

pipeline = DataPreparationPipeline(config)
result = pipeline.run()
print(f"n_new_sessions={result.n_new_sessions}, n_skipped_sessions={result.n_skipped_sessions}")
print(f"long_formats_saved={result.long_formats_saved}")
print(f"n_wide_features={len(result.wide_features_generated)}")

n_new_sessions=0, n_skipped_sessions=46
long_formats_saved=['anatomical', 'diffusion_AMICONODDI', 'diffusion_DIPYDKI', 'diffusion_DIPYMAPMRI', 'diffusion_DSIStudio', 'diffusion_MRtrix3actHSVS']
n_wide_features=1278
